<h1><center>Analyse de conversations patient-thérapeute sur la santé mentale</h1>

<p><center>La santé mentale est un enjeu majeur de la santé publique. Ce projet vise, dans un pre-
mier temps, à exploiter le jeu de données pour comprendre les types de conversations et leur
structure, puis à analyser les sentiments des échanges afin de détecter les émotions et états
psychologiques des patients. Ensuite, il se concentre sur l’accès et le traitement des conversa-
tions, en résumant les échanges et en identifiant les schémas linguistiques récurrents. Enfin, il
prévoit le développement d’un système question-réponse, capable d’automatiser les interactions
et fournir des réponses.</p>

<center><nav>
    <a href="https://www.kaggle.com/datasets/thedevastator/nlp-mental-health-conversations">Données</a> |
    <a href="https://github.com/cypnet/Analyse-de-conversations-sur-la-sant-mentale-/tree/Logan">Github</a>
</nav>

In [1]:
# Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_path = './train.csv'
df = pd.read_csv(data_path)
print(f"Détails nombre de lignes/colonnes: {df.shape}")
print(f"\nDétails des colonnes: {df.columns.tolist()}")
print(f"\nAperçu des premières lignes:")
df.head()

Détails nombre de lignes/colonnes: (3512, 2)

Détails des colonnes: ['Context', 'Response']

Aperçu des premières lignes:


,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


<h3>Nettoyage des données (pré-traitement)</h3>

<h4>Cellules vides</h4> 

In [3]:
# On doit d'abord vérifier si il y a des cellules vides pour les deux colonnes
print("\nCellules vides (avant traitement): \n",df.isnull().sum())
df = df.dropna(subset=['Response', 'Context'])
print("\nCellules vides (après traitement): \n",df.isnull().sum())


Cellules vides (avant traitement): 
 Context     0
Response    4
dtype: int64

Cellules vides (après traitement): 
 Context     0
Response    0
dtype: int64


<h4>Doublons de cellules</h4>

In [4]:
# On vérifie ensuite si il y a des doublons de cellules
print("\nCellules dupliquées (avant traitement):",df.duplicated().sum())
df = df.drop_duplicates()
print("\nCellules dupliquées (après traitement):",df.duplicated().sum())


Cellules dupliquées (avant traitement): 760

Cellules dupliquées (après traitement): 0


<h4>Ponctuations</h4>
<p>Cela nous permettra de trier les mots plus facilement plus tard, puisque les discussions sont faites de façon orale, les trier n'ont pas vraiment d'utilités</p>

In [5]:
# Fonction qu'on va utiliser pour compter les ponctuations
def count_ponctuation(txt):
    txt = str(txt) # conversion en String
    ponct_counts = {
        'point': txt.count('.'),
        'virgule': txt.count(','),
        'deux points': txt.count(':'),
        'point virgule': txt.count(';'),
        'interogation': txt.count('?'),
        'exclamation': txt.count('!'),
        'guillemet': txt.count('"') + txt.count("'"),
        'tiret': txt.count('-'),
        'slash': txt.count('/')
    }
    return ponct_counts

ponct_data = df['Response'].apply(count_ponctuation)
ponct_df = pd.DataFrame(ponct_data.tolist()) 

print(f"\nPonctuation totale (Avant traitement):\n{ponct_df.sum().sort_values(ascending=False)} \n")

# Pour chaque cellules, on supprimme les ponctuations qui sont comprises dans count_ponctuation en les remplaçant par ''
def remove_ponctuation(txt):
    txt = str(txt) # conversion en String
    ponctuations = ['.', ',', ':', ';', '?', '!', '"', "'", '-','/']
    for p in ponctuations:
        txt = txt.replace(p, '')
    return txt

# On supprime les ponctuations
df['Response'] = df['Response'].apply(remove_ponctuation)
df['Context'] = df['Context'].apply(remove_ponctuation)

ponct_data = df['Response'].apply(count_ponctuation)
ponct_df = pd.DataFrame(ponct_data.tolist()) 

print(f"\nPonctuation totale (Après traitement):\n{ponct_df.sum().sort_values(ascending=False)} \n")


Ponctuation totale (Avant traitement):
point            26633
virgule          19425
guillemet        10001
slash             2728
tiret             2648
interogation      2404
exclamation       1105
deux points        872
point virgule      449
dtype: int64 


Ponctuation totale (Après traitement):
point            0
virgule          0
deux points      0
point virgule    0
interogation     0
exclamation      0
guillemet        0
tiret            0
slash            0
dtype: int64 



In [6]:
# On vérifie que toute les cellules soient du type String et si ce n'est pas le cas (NaN) les remplacer par ''
df['Response'] = df['Response'].fillna('').astype(str)

# On regarde si les cellules Response et Context sont unique, si ce n'est pas le cas alors on supprimme les dupliquées
df = df.drop_duplicates(subset=['Response', 'Context'])

# Pour chaque cellules, on supprimme les ponctuations qui sont comprises dans count_ponctuation en les remplaçant par ''
def remove_ponctuation(txt):
    txt = str(txt) # conversion en String
    ponctuations = ['.', ',', ':', ';', '?', '!', '"', "'", '-','/']
    for p in ponctuations:
        txt = txt.replace(p, '')
    return txt

# On supprime les ponctuations
df['Response'] = df['Response'].apply(remove_ponctuation)
df['Context'] = df['Context'].apply(remove_ponctuation)

<h4>Passer tout les mots en minuscules</h4>
<p>Transformer tout les mots en minuscules nous permettra par la suite de les traiter plus facilement et éviter des situations telle que "toto" soit différent de "Toto".</p>

In [7]:
# compter les majuscules AVANT traitement
df['Response_upper'] = df['Response'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
df['Context_upper'] = df['Context'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

print("\nNombre de majuscules dans Response (avant):", df['Response_upper'].sum())
print("Nombre de majuscules dans Context (avant):", df['Context_upper'].sum())

# transformer le texte en minuscules
df['Response'] = df['Response'].str.lower()
df['Context'] = df['Context'].str.lower()


# compter les majuscules APRES traitement
df['Response_upper_after'] = df['Response'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
df['Context_upper_after'] = df['Context'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

print("\nNombre de majuscules dans Response (après):", df['Response_upper_after'].sum())
print("Nombre de majuscules dans Context (après):", df['Context_upper_after'].sum())


Nombre de majuscules dans Response (avant): 61125
Nombre de majuscules dans Context (avant): 20485

Nombre de majuscules dans Response (après): 0
Nombre de majuscules dans Context (après): 0


<h4>Tokenisation /!\ À CORRIGER BUG DE TOKENISATION /!\</h4>
<p>On va maitenant "découper" les mots de chaque cellules en <i>tokens</i>.<br>
Pour pouvoir les définir, on va utiliser la librairie <b>spaCy</b> qui est une version un peut plus moderne que nltk (The Natural Language Toolkit) </p>

<p><i>Il faut au préalable télécharger le modèle pour pouvoir l'utiliser:</i> <code>python3 -m spacy download en_core_web_sm
</code></p>

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm") # On charge spaCy dans python

<h5>Attention !</h5>
<p>Ici on traite une grande quantité de données, tokeniser (si ça se dit...) les mots bêtements va prendre énormément de temps, puisque spacy va devoir exécuter tokenize sur chaque ligne </p>
<p> On va devoir utiliser <code>nlp.pipe()</code> pour traiter les cellules en batch</p>

In [10]:
# Tokens pour la colonne Response
# Response
resp_docs = list(nlp.pipe(df["Response"].astype(str), batch_size=200))
df["Response_tokens"] = [[token.text for token in doc] for doc in resp_docs]

# Context
cont_docs = list(nlp.pipe(df["Context"].astype(str), batch_size=200))
df["Context_tokens"] = [[token.text for token in doc] for doc in cont_docs]


In [11]:
# On affiche le nombre de token total
total_tokens = df['Response_tokens'].apply(len).sum() + df['Context_tokens'].apply(len).sum()
print(f"\nNombre total de tokens: {total_tokens}")

# On affiche les 5 tokens les plus fréquents dans Response et Context pour avoir un apperçu
from collections import Counter
resp_tokens = [token for tokens in df['Response_tokens'] for token in tokens]
cont_tokens = [token for tokens in df['Context_tokens'] for token in tokens]
print(f"\n5 tokens les plus fréquents dans Response: {Counter(resp_tokens).most_common(5)}")
print(f"5 tokens les plus fréquents dans Context: {Counter(cont_tokens).most_common(5)}")


Nombre total de tokens: 671134

5 tokens les plus fréquents dans Response: [('you', 22082), ('to', 20165), ('the', 13244), ('and', 12961), ('a', 11199)]
5 tokens les plus fréquents dans Context: [('i', 12831), ('and', 5396), ('to', 5177), ('my', 3825), ('a', 3486)]


<p><i>On remarque que la plupart des mots les plus utilisés ne sont que des mots dit "vide"...</i></p>

<h4>Lemmatisation</h4>
<p>Avant de traiter ses stopwords, il nous faut d'abords transformer les mots (tokens) en leurs forme de base, par exemple "running" deviendra "run", ou bien "better" deviendra "good". <br>
Cela permettra d'une part de diminuer drastiquement le nombre de tokens mais aussi de regrouper des mots qui sont similaires</p>

In [12]:
# On va faire la lemmatisation sur les tokens de Response et Context
df["Response_lemmas"] = [[token.lemma_ for token in doc] for doc in resp_docs]
df["Context_lemmas"] = [[token.lemma_ for token in doc] for doc in cont_docs]

In [15]:
# On affiche maintenant le nombre de tokens après lemmatisation
total_lemmas = df['Response_lemmas'].apply(len).sum() + df['Context_lemmas'].apply(len).sum()
print(f"\nNombre total de tokens après lemmatisation: {total_lemmas}")


Nombre total de tokens après lemmatisation: 671134


<h4>Suppression des StopWords</h4>
<p>Des <i>stopwords</i> (ou mots vides en français) sont des mots n'ayant pas de réelle significations. Ses mots sont souvents des adverbes, des pronoms ou encore des mots de liaisons.<br>
<i>source: <a href="https://fr.wikipedia.org/wiki/Mot_vide">Wikipedia: Mot Vide</a> </i> </p>

In [16]:
# On récupère la liste des stopwords de spaCy
stopwords = nlp.Defaults.stop_words

# On compte le nombre de stopwords dans Response et Context
df['Response_stopwords'] = df['Response_tokens'].apply(
    lambda tokens: sum(1 for token in tokens if token in stopwords)
)

df['Context_stopwords'] = df['Context_tokens'].apply(
    lambda tokens: sum(1 for token in tokens if token in stopwords)
)

# Fonction qui retire les stopwords d'une liste de tokens
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stopwords]

# On retire les stopwords des tokens de Response et Context
df['Response_tokens_no_stop'] = df['Response_tokens'].apply(remove_stopwords)
df['Context_tokens_no_stop'] = df['Context_tokens'].apply(remove_stopwords)

In [20]:
# On compte le nombre de tokens sans stopwords dans Response et Context
total_tokens_no_stop = df['Response_tokens_no_stop'].apply(len).sum() + df['Context_tokens_no_stop'].apply(len).sum()
print(f"Nombre de stopwords dans Response et Context: {total_tokens - total_tokens_no_stop}, soit {((total_tokens - total_tokens_no_stop) / total_tokens) * 100:.2f}% du total des tokens")
print(f"\nNombre total de tokens sans stopwords: {total_tokens_no_stop}")

Nombre de stopwords dans Response et Context: 396074, soit 59.02% du total des tokens

Nombre total de tokens sans stopwords: 275060


<h3>Analyse Lexicale</h3>

In [ ]:
# TODO

<h3>Modélisation et extraction de sujets (Topic Modeling)</h3>